# 4. Feature definition: feature matrix <a id="4"></a>
In this section we will featurise the kinase domains in our dataset and select only features of statistical relevance.


## Table of contents

- [4.1 Feature matrix](#41)
  - [Cα distance counterpart](#c-distance-counterpart)


## Backend map

How this notebook connects to `workflow/` modules:

```mermaid
flowchart LR
  nb["09-FeatureMatrix"]
  m0["workflow.analyse_alignment_foldmason"]
  nb --> m0
  m1["workflow.feature_selection"]
  nb --> m1
  m2["workflow.utilities"]
  nb --> m2
```


![State of the workflow](images/FeatureSelection.png)

In this notebook, we will be focusing on the second step: feature selection.

To get started, let's load some packages!

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd

from workflow.analyse_alignment_foldmason import analyse_alignment
from workflow.feature_selection import FeatureSelection
from workflow.utilities import PDBDownloader
from workflow.utilities import count_pdb_files, braf_res, clear_and_make, make_seg, copy_filtered_pdbs


## 4.1 Feature matrix  <a id="41"></a>
Here we define our featurisation approach. We construct a feature matrix for each structure in our dataset.

Let's first import and initialise the class `FeatureSelection()` that we use to perform all the necessary steps to featurise kinase domains outside the activation loop region.

In [ ]:
from workflow.feature_selection import FeatureSelection

# Initialize the feature selection object
fs = FeatureSelection(
    dfg_index=145,  # Position of DFG motif
    ape_index=174,  # Position of APE motif
    conservation_threshold=0.70  # 70% conservation threshold
)

print("FeatureSelection initialized")
print(f"Activation loop region: {fs.dfg_index} to {fs.ape_index}")
print(f"Conservation threshold: {fs.conservation_threshold * 100}%")

The first step uses the conservation data to identify residues that are:
- Conserved in ≥70% of structures
- Located outside the activation loop region (DFG to APE motif)

Conservation is loaded from `Results/activation_segments/multi_aligned_foldmason/conservation.npy` (or recomputed from `msa_3di.fa` if needed), so this cell does not depend on an earlier notebook `conservation` variable.


<div class="alert alert-warning">
<b>PRE-REQUISITE:</b> You must first run the conservation analysis cell which calculates the <code>conservation</code> variable by analyzing the multi-structure alignment.
</div>

In [ ]:
import os
import numpy as np

# Load reference residue names
reference_residues = braf_res()

# Resolve conservation independently of earlier notebook cells:
# 1) reuse in-memory `conservation` if present
# 2) else load cached .npy from FoldMason analysis
# 3) else recompute from existing msa_3di.fa
CONSERVATION_NPY = "Results/activation_segments/multi_aligned_foldmason/conservation.npy"
ALIGNMENT_FILE = "Results/activation_segments/multi_aligned_foldmason/msa_3di.fa"

if "conservation" in globals() and conservation is not None:
    conservation = np.asarray(conservation, dtype=float)
    print(f"Using in-memory conservation array (len={len(conservation)})")
elif os.path.exists(CONSERVATION_NPY):
    conservation = np.load(CONSERVATION_NPY)
    print(f"Loaded conservation from cache: {CONSERVATION_NPY} (len={len(conservation)})")
else:
    if not os.path.exists(ALIGNMENT_FILE):
        raise FileNotFoundError(
            f"Missing conservation cache ({CONSERVATION_NPY}) and alignment file "
            f"({ALIGNMENT_FILE}). Run FoldMason conservation (e.g. in 04b/08a) first."
        )
    from workflow.analyse_alignment_foldmason import analyse_alignment

    print(f"Recomputing conservation from {ALIGNMENT_FILE}...")
    analyser = analyse_alignment()
    conservation_run = analyser.run_multi_alignment_conservation_analysis(
        alignment_file=ALIGNMENT_FILE,
        reference_name="6UAN_chainD",
        reference_residues=reference_residues,
        conservation_threshold=0.70,
        output_plot="Results/multi_alignment_foldMason_conservation.png",
        output_csv="Results/conserved_residues_70percent.csv",
        show_plot=False,
        verbose=False,
    )
    conservation = np.asarray(conservation_run["conservation"], dtype=float)
    os.makedirs(os.path.dirname(CONSERVATION_NPY), exist_ok=True)
    np.save(CONSERVATION_NPY, conservation)
    print(f"Saved conservation cache: {CONSERVATION_NPY} (len={len(conservation)})")

# Identify conserved residues (uses 70% threshold set in FeatureSelection initialization)
# This will find residues that are:
# - Conserved in ≥70% of structures
# - Located OUTSIDE the activation loop (not between DFG at 145 and APE at 174)
fs.identify_conserved_residues(
    conservation=conservation,
    reference_residues=reference_residues
)

print(f"\nFound {len(fs.fully_conserved)} conserved residues (≥70% conservation)")


### Cα distance counterpart <a id="c-distance-counterpart"></a>

Alongside side-chain minimum distances we compute **Cα–Cα Euclidean distances** for the same conserved residue pairs.  Each pair yields a single, unambiguous distance (one Cα per residue).  Structures in which a conserved residue is absent (alignment gap or missing atoms) will have NaN for the affected pair; median imputation is applied during feature-matrix construction exactly as for side-chain features.

A separate `FeatureSelection` object (`fs_ca`) is used to hold the Cα feature data so that the side-chain pipeline on `fs` is not affected.


In [ ]:
from workflow.feature_selection import FeatureSelection

# Create a FeatureSelection object for Cα features, sharing the same conserved-residue list
fs_ca = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.70)
fs_ca.fully_conserved = fs.fully_conserved  # reuse conservation analysis from fs
print(f"fs_ca ready.  Conserved residues: {len(fs_ca.fully_conserved)}")

Before calculating distance features, we need to organize structures into cluster-specific folders based on the PCA clustering results in order to be able to easily retain the label of each structure in our dataset.

In [ ]:
# Load PCA cluster labels (use hierarchical clustering results)
pca_labels_file = 'cluster_labels_my_analysis_hierarchical.txt'
df_labels = pd.read_csv(pca_labels_file, skiprows=1, header=None, 
                        names=['ClusterLabel', 'PDBCode', 'FullName'])

# Create output directories for each cluster (clearing old structures)
cluster0_dir = "Results/activation_segments/structuresToFeaturiseCluster0/"
cluster1_dir = "Results/activation_segments/structuresToFeaturiseCluster1/"
clear_and_make(cluster0_dir)
clear_and_make(cluster1_dir)

# Source directory with InterPro protein chains (PDBID_CHAIN.pdb)
source_dir = "Results/InterPro_protein_chains/"

# Organize structures by cluster
copied_cluster0 = []
copied_cluster1 = []
missing_files = []

for _, row in df_labels.iterrows():
    cluster_label = int(row['ClusterLabel'])
    pdb_basename = f"{row['PDBCode']}.pdb"
    source_file = os.path.join(source_dir, pdb_basename)

    if not os.path.isfile(source_file):
        missing_files.append(pdb_basename)
        continue

    if cluster_label == 0:
        dest_file = os.path.join(cluster0_dir, pdb_basename)
        shutil.copy2(source_file, dest_file)
        copied_cluster0.append(pdb_basename)
    elif cluster_label == 1:
        dest_file = os.path.join(cluster1_dir, pdb_basename)
        shutil.copy2(source_file, dest_file)
        copied_cluster1.append(pdb_basename)

print("=== PCA Cluster Organization ===")
print(f"Matching InterPro chains by PDBCode (e.g. 1A9U_A → 1A9U_A.pdb)")
print(f"Source directory: {source_dir}")
print(f"\nCluster 0: {len(copied_cluster0)} structures → {cluster0_dir}")
print(f"Cluster 1: {len(copied_cluster1)} structures → {cluster1_dir}")
print(f"Total structures organized: {len(copied_cluster0) + len(copied_cluster1)}")

if missing_files:
    print(f"\n⚠️  Warning: {len(missing_files)} files not found in {source_dir}")
    print("First 5 missing files:")
    for f in missing_files[:5]:
        print(f"  - {f}")


We can now calculate pairwise distances between conserved residues for structures in each cluster which will constitute our feature dataset.

Alignment objects are loaded from the FoldMason multi-alignment (`msa_3di.fa`) if `multi_data` is not already in memory.


We calculate distances first for the structures in cluster 0.

In [ ]:
import os
from workflow.analyse_alignment_foldmason import analyse_alignment

ALIGNMENT_FILE = "Results/activation_segments/multi_aligned_foldmason/msa_3di.fa"
REFERENCE_NAME = "6UAN_chainD"

# Resolve multi-alignment independently of earlier notebook cells
if "multi_data" in globals() and multi_data is not None and "structures" in multi_data:
    print(f"Using in-memory multi_data ({len(multi_data['structures'])} structures)")
else:
    if not os.path.exists(ALIGNMENT_FILE):
        raise FileNotFoundError(
            f"Missing FoldMason alignment: {ALIGNMENT_FILE}. "
            "Run FoldMason multi-alignment (e.g. in 04b/08a) first."
        )
    print(f"Loading multi_data from {ALIGNMENT_FILE}...")
    _analyser = analyse_alignment()
    multi_data = _analyser.load_multi_alignment(
        ALIGNMENT_FILE,
        reference_name=REFERENCE_NAME,
    )
    if multi_data is None:
        raise RuntimeError(f"Failed to load multi-alignment from {ALIGNMENT_FILE}")
    print(f"Loaded {len(multi_data['structures'])} structures vs reference")

aligned_structures = multi_data["structures"]

# Ensure cluster dirs exist (from previous cell)
if "cluster0_dir" not in globals():
    cluster0_dir = "Results/activation_segments/structuresToFeaturiseCluster0/"
if "cluster1_dir" not in globals():
    cluster1_dir = "Results/activation_segments/structuresToFeaturiseCluster1/"

# Choose which cluster to analyze
# Cluster 0 is typically the inactive state, Cluster 1 is active state (or vice versa)
pdb_directory = cluster0_dir

if not os.path.isdir(pdb_directory):
    raise FileNotFoundError(
        f"Cluster directory not found: {pdb_directory}. "
        "Run the PCA cluster organization cell above first."
    )

print(f"Analyzing structures from: {pdb_directory}")
print(f"Number of structures: {len([f for f in os.listdir(pdb_directory) if f.endswith('.pdb')])}")

# Calculate distances between conserved residues
distance_df_cluster0 = fs.calculate_intra_structure_distances(
    aligned_structures=aligned_structures,
    pdb_directory=pdb_directory,
    alignment_function=make_seg,
)

print(f"\n=== Distance Calculation Results (Cluster 0) ===")
print(f"Calculated {len(distance_df_cluster0)} distance measurements")
print(f"Across {len(fs.structures)} structures")
print(f"\nFirst few rows:")
distance_df_cluster0.head()


In [ ]:
# Cα distances for Cluster 0
# fs.structures was just populated by the side-chain call above — reuse it (no PDB re-read)
ca_distance_df_cluster0 = fs.calculate_intra_structure_ca_distances()
print(f"\n=== Cα Distance Results (Cluster 0) ===")
print(f"Measurements: {len(ca_distance_df_cluster0)}")
print(f"Structures:   {ca_distance_df_cluster0['structure'].nunique()}")

Then we calculate distances for structures in cluster 1.

In [ ]:
# Calculate distances for Cluster 1
if "aligned_structures" not in globals():
    raise NameError(
        "aligned_structures is not defined. Run the Cluster 0 distance cell above first "
        "(it loads multi_data / aligned_structures)."
    )

if "cluster1_dir" not in globals():
    cluster1_dir = "Results/activation_segments/structuresToFeaturiseCluster1/"

pdb_directory_cluster1 = cluster1_dir

if not os.path.isdir(pdb_directory_cluster1):
    raise FileNotFoundError(
        f"Cluster directory not found: {pdb_directory_cluster1}. "
        "Run the PCA cluster organization cell above first."
    )

print(f"Analyzing structures from: {pdb_directory_cluster1}")
print(f"Number of structures: {len([f for f in os.listdir(pdb_directory_cluster1) if f.endswith('.pdb')])}")

# Calculate distances between conserved residues for Cluster 1
distance_df_cluster1 = fs.calculate_intra_structure_distances(
    aligned_structures=aligned_structures,
    pdb_directory=pdb_directory_cluster1,
    alignment_function=make_seg,
)

print(f"\n=== Distance Calculation Results (Cluster 1) ===")
print(f"Calculated {len(distance_df_cluster1)} distance measurements")
print(f"Across {len(fs.structures)} structures")
print(f"\nFirst few rows:")
distance_df_cluster1.head()


In [ ]:
# Cα distances for Cluster 1 — reuse fs.structures just loaded above
ca_distance_df_cluster1 = fs.calculate_intra_structure_ca_distances()
print(f"\n=== Cα Distance Results (Cluster 1) ===")
print(f"Measurements: {len(ca_distance_df_cluster1)}")
print(f"Structures:   {ca_distance_df_cluster1['structure'].nunique()}")

Now we can create a dataset of all the features extracted from our kinase domains which will be later used in classification.

In [ ]:
# Combine distance measurements from both clusters
combined_distance_df = pd.concat([distance_df_cluster0, distance_df_cluster1], 
                                  ignore_index=True)

# Update fs.intra_structure_df with combined data
fs.intra_structure_df = combined_distance_df

print(f"=== Combined Distance Data ===")
print(f"Cluster 0: {len(distance_df_cluster0)} measurements from {distance_df_cluster0['structure'].nunique()} structures")
print(f"Cluster 1: {len(distance_df_cluster1)} measurements from {distance_df_cluster1['structure'].nunique()} structures")
print(f"Combined: {len(combined_distance_df)} total measurements from {combined_distance_df['structure'].nunique()} structures")
print(f"\nSample of combined data:")
print(combined_distance_df.head())


In [ ]:
# Combine Cα distance DataFrames from both clusters
combined_ca_distance_df = pd.concat(
    [ca_distance_df_cluster0, ca_distance_df_cluster1], ignore_index=True
)
# Hand the combined Cα distance table to fs_ca
fs_ca.intra_structure_df = combined_ca_distance_df

print(f"=== Combined Cα Distance Data ===")
print(f"Cluster 0: {len(ca_distance_df_cluster0)} measurements, "
      f"{ca_distance_df_cluster0['structure'].nunique()} structures")
print(f"Cluster 1: {len(ca_distance_df_cluster1)} measurements, "
      f"{ca_distance_df_cluster1['structure'].nunique()} structures")
print(f"Combined : {len(combined_ca_distance_df)} measurements, "
      f"{combined_ca_distance_df['structure'].nunique()} structures")

Let's add a column to our feature dataset in order to be able to track what labels are associated with what structures.


In [ ]:
# Check if cluster directories are defined (from Cell 105)
try:
    cluster0_dir
    cluster1_dir
except NameError:
    # If not defined, set them to the expected paths
    print("⚠️  WARNING: cluster0_dir and cluster1_dir not found in environment.")
    print("Please run Cell 105 first to organize structures by PCA clusters.")
    print("Using default paths as fallback...\n")
    
    cluster0_dir = "Results/activation_segments/structuresToFeaturiseCluster0/"
    cluster1_dir = "Results/activation_segments/structuresToFeaturiseCluster1/"

# Use the cluster directories created from PCA analysis
cluster_dirs = {
    0: cluster0_dir,  # "Results/activation_segments/structuresToFeaturiseCluster0/"
    1: cluster1_dir   # "Results/activation_segments/structuresToFeaturiseCluster1/"
}

print("Using PCA-based cluster directories:")
print(f"  Cluster 0: {cluster_dirs[0]}")
print(f"  Cluster 1: {cluster_dirs[1]}")

# Assign labels based on cluster membership
fs.assign_labels_from_clusters(cluster_dirs)

# Check label distribution
print("\n=== Label Distribution in Dataset ===")
if fs.intra_structure_df is not None and 'label' in fs.intra_structure_df.columns:
    # IMPORTANT: Each structure has MANY rows (one per residue pair distance)
    # So we need to count both rows AND unique structures
    
    print("📊 Unique structures per label:")
    for label in sorted(fs.intra_structure_df['label'].unique()):
        if label == -1:
            continue
        n_structures = fs.intra_structure_df[fs.intra_structure_df['label'] == label]['structure'].nunique()
        n_measurements = len(fs.intra_structure_df[fs.intra_structure_df['label'] == label])
        print(f"  Label {label}: {n_structures} structures ({n_measurements} distance measurements)")
    
    total_unique = fs.intra_structure_df[fs.intra_structure_df['label'] != -1]['structure'].nunique()
    total_measurements = len(fs.intra_structure_df[fs.intra_structure_df['label'] != -1])
    print(f"\n✅ Total: {total_unique} labeled structures, {total_measurements} total distance measurements")
    
    if (fs.intra_structure_df['label'] == -1).any():
        unlabeled = fs.intra_structure_df[fs.intra_structure_df['label'] == -1]['structure'].nunique()
        print(f"⚠️  Warning: {unlabeled} structures without labels")
else:
    print("No labels assigned yet. Run distance calculation and combination first.")

In [ ]:
# Assign cluster labels to fs_ca (same cluster directories used for side-chain labelling)
fs_ca.assign_labels_from_clusters(cluster_dirs)
print("\n=== Cα Label Distribution ===")
if fs_ca.intra_structure_df is not None and 'label' in fs_ca.intra_structure_df.columns:
    for lbl in sorted(fs_ca.intra_structure_df['label'].unique()):
        if lbl == -1:
            continue
        n = fs_ca.intra_structure_df[fs_ca.intra_structure_df['label'] == lbl]['structure'].nunique()
        print(f"  Label {lbl}: {n} structures")

We can now organise all distance measurements into a structured feature matrix for each structure. Since not all residues between which we compute distances are conserved we use median imputation to ensure all feature matrices include the same number of features.

In [ ]:
# Build feature matrix with median imputation
feature_matrix, imputation_mask = fs.build_feature_matrix(
    use_median_imputation=True
)

print(f"\nFeature matrix shape: {feature_matrix.shape}")
print(f"Number of structures: {len(fs.structure_names)}")
print(f"Number of features: {len(fs.unique_pairs)}")
print(f"\nSample feature names: {[f'{p[0]}-{p[1]}' for p in fs.unique_pairs[:5]]}")

# Save all results for later reloading
print("\n" + "="*60)
print("Saving feature matrix and related data...")
print("="*60)
fs.save_results(output_prefix="")
print("✅ All data saved! Can be reloaded without recomputing.")

In [ ]:
# Build Cα feature matrix (with median imputation for any missing Cα atoms)
ca_feature_matrix, ca_imputation_mask = fs_ca.build_feature_matrix(use_median_imputation=True)
print(f"\nCα feature matrix shape: {ca_feature_matrix.shape}")
print(f"Structures: {len(fs_ca.structure_names)}")
print(f"Feature pairs: {len(fs_ca.unique_pairs)}")

# Save Cα features using prefix 'ca_'
print("\n" + "="*60)
print("Saving Cα feature matrix ...")
print("="*60)
fs_ca.save_results(output_prefix="ca_")
print("✅ Cα features saved (prefix: 'ca_')")

Let's visualise some feature matrices as heat maps.

In [ ]:
# Plot example heatmaps
fs.plot_distance_heatmaps(
    n_examples=4,
    save_dir=None  # Set to a directory path to save all heatmaps
)